# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** *64*  
**Kaggle challenge:** *Classic*   
**Kaggle team name (exact):** "*Team Nah!*"  

**Author 1 (sciper):** Andrew Brown (370751)  
**Author 2 (sciper):** Nour Lachat (302397)   
**Author 3 (sciper):** Henry Farrell (402247) 

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

> Your comments  
> ...

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

In [ ]:
# Import main packages
from skimage.morphology import remove_small_objects, remove_small_holes, closing, disk, opening
from skimage.transform import rotate, resize
from sklearn.metrics.pairwise import euclidean_distances
from skimage.measure import regionprops

import cv2
import numpy as np

In [ ]:
image_folder='chocolate-recognition-classic/dataset_project_iapr2025/test'
# Loop through filenames from L1000777.jpg to L1000790.jpg
for i in range(1000757, 1000758):
    filename = f'L{i}.JPG'
    img_path = os.path.join(image_folder, filename)
    
    if os.path.exists(img_path):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert from BGR to RGB for matplotlib
        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.title(filename)
        plt.axis('off')
        plt.show()
    else:
        print(f"Image {filename} not found.")

In [ ]:
def find_contour(images: np.ndarray):
    """
    Find the contours for the set of images
    
    Args
    ----
    images: np.ndarray (N, 28, 28)
        Source images to process

    Return
    ------
    contours: list of np.ndarray
        List of N arrays containing the coordinates of the contour. Each element of the 
        list is an array of 2d coordinates (K, 2) where K depends on the number of elements 
        that form the contour. 
    """

    # Get number of images to process
    N, _, _ = np.shape(images)
    # Fill in dummy values (fake points)
    contours = [np.array([[0, 0], [1, 1]]) for i in range(N)]
    contour= [np.array([[0, 0], [1, 1]]) for i in range(N)]

    # ------------------
    for i in range(N):
        contours_img=cv2.findContours(images[i], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        max_length = 0
        longest_contour = None

        for j in range(len(contours_img[0])):
            length=len(contours_img[0][j])
            if length > max_length:
                longest_contour = contours_img[0][j]
                max_length = length
        contours[i] = longest_contour

        if len(contours_img[0]) > 0:
            for j in range(len(contours_img[0])):
                length = len(contours_img[0][j])
                if length > max_length:
                    longest_contour = contours_img[0][j]
                    max_length = length
            
            # Only update contours[i] and squeeze if a contour was found
            if longest_contour is not None:
                contours[i] = longest_contour
                contours[i] = contours[i].squeeze()  # Now shape is (K, 2)

        if longest_contour is not None:
            contours[i] = longest_contour
            contours[i] = contours[i].squeeze()  # Now shape is (K, 2)
        else:

            print(f"Warning: No valid contour found for image {i}")
      # ------------------
    return contours

In [ ]:
def find_contour(image: np.ndarray):
    """
    Find the contours for a single image.

    Args
    ----
    image: np.ndarray (H, W)
        Source image to process (binary image).

    Return
    ------
    contours: list of np.ndarray
        List of arrays containing the coordinates of the contours. Each element of the 
        list is an array of 2D coordinates (K, 2) where K depends on the number of elements 
        that form the contour.
    """
    # Ensure the input image is binary
    binary_image = (image > 0).astype(np.uint8)

    # Find contours using OpenCV
    contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Convert contours to a list of arrays with shape (K, 2)
    contours = [contour.squeeze() for contour in contours if contour.size > 0]

    return contours

In [ ]:
import cv2
import numpy as np

def simple_segmentation(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Thresholds for neutral/light backgrounds — adjust as needed
    lower = np.array([0, 0, 160])
    upper = np.array([180, 40, 255])

    # Create binary mask: 0 for background, 255 for foreground
    background_mask = cv2.inRange(hsv, lower, upper)
    foreground_mask = cv2.bitwise_not(background_mask)

    # Optional: morphological cleanup
    kernel = np.ones((5, 5), np.uint8)
    cleaned = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, kernel)

    return cleaned

# Load image
image_path = 'chocolate-recognition-classic/dataset_project_iapr2025/test/L1000757.JPG'  # Replace with your image path
image = cv2.imread(image_path)
if image is None:
    raise FileNotFoundError(f"Image not found at {image_path}")

# Create mask
mask = simple_segmentation(image)

# Create a blue background (same shape as image)
blue_background = np.full_like(image, (255, 0, 0))  # BGR for blue

# Combine chocolates with blue background
foreground = cv2.bitwise_and(image, image, mask=mask)
background = cv2.bitwise_and(blue_background, blue_background, mask=cv2.bitwise_not(mask))
result = cv2.add(foreground, background)

# Convert to RGB for display
original_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

# Display side-by-side
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(original_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(result_rgb)
plt.title("Background Replaced with Blue")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def get_binary_mask(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Threshold for neutral/light backgrounds
    lower = np.array([0, 0, 160])
    upper = np.array([180, 40, 255])

    background_mask = cv2.inRange(hsv, lower, upper)
    foreground_mask = cv2.bitwise_not(background_mask)

    # Morphological cleanup (optional)
    kernel = np.ones((5, 5), np.uint8)
    cleaned = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, kernel)

    # Convert to binary: 0 = foreground (chocolates), 1 = background
    binary_image = np.where(cleaned > 0, 0, 1).astype(np.uint8)

    return binary_image

# Load image
image_path = 'chocolate-recognition-classic/dataset_project_iapr2025/test/L1000757.JPG'  # Replace with your image path
image = cv2.imread(image_path)
if image is None:
    raise FileNotFoundError("Image not found.")

binary_mask = get_binary_mask(image)

# Display binary mask
plt.imshow(binary_mask, cmap='gray')
plt.title("Binary Image (0=Object, 1=Background)")
plt.axis('off')
plt.show()

In [ ]:
image_path = 'chocolate-recognition-classic/dataset_project_iapr2025/test/L1000757.JPG'  # Replace with your image path
image = cv2.imread(image_path)
mask = simple_segmentation(image)
result = cv2.bitwise_and(image, image, mask=mask)

In [ ]:
image_path = 'chocolate-recognition-classic/dataset_project_iapr2025/test/L1000757.JPG'  # Replace with your image path
image = cv2.imread(image_path)
# Convert to grayscale and preprocess (e.g., thresholding)
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
_, binary_image = cv2.threshold(gray_image, 127, 255, cv2.THRESH_BINARY)

# Find contours
contours = find_contour(binary_image)

# Draw contours in red on the original image
output_image = image.copy()
cv2.drawContours(output_image, contours, -1, (0, 0, 255), 2)  # Red color (BGR: (0, 0, 255)), thickness=2

# Display the result
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB))  # Convert BGR to RGB for matplotlib
plt.title("Contours in Red")
plt.axis('off')
plt.show()